# Tests de integración de `get_trips_from_flow()`

Notebook orientado a verificar el comportamiento end-to-end de OP-13 `Get trips from flows`,
considerando resultado tabular, resultado operacional y trazabilidad.

## Bloque 0: Setup de fixtures para integración

### 0.1 - Imports y utilidades base

Qué prepara: imports del notebook, helpers pequeños para H3 y snapshots de estado de entrada.

In [1]:
from copy import deepcopy

import pandas as pd
import h3

from pylondrina.datasets import FlowDataset, TripDataset
from pylondrina.errors import PylondrinaError
from pylondrina.queries.flows import get_trips_from_flows
from pylondrina.schema import TripSchema


def h3_from_latlon(lat: float, lon: float, res: int = 8) -> str:
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lon, res)
    return h3.geo_to_h3(lat, lon, res)


def h3_to_parent(cell: str, res: int) -> str:
    if hasattr(h3, "cell_to_parent"):
        return h3.cell_to_parent(cell, res)
    return h3.h3_to_parent(cell, res)


def assert_issue_codes(issues, expected_codes):
    observed = [issue.code for issue in issues]
    assert observed == expected_codes, f"Esperados={expected_codes}, observados={observed}"


def snapshot_flowdataset_state(flows: FlowDataset) -> dict:
    return {
        "flows": flows.flows.copy(deep=True),
        "flow_to_trips": None if flows.flow_to_trips is None else flows.flow_to_trips.copy(deep=True),
        "aggregation_spec": deepcopy(flows.aggregation_spec),
        "metadata": deepcopy(flows.metadata),
        "provenance": deepcopy(flows.provenance),
        "source_trips": flows.source_trips,
    }


def snapshot_tripdataset_state(trips: TripDataset) -> dict:
    return {
        "data": trips.data.copy(deep=True),
        "metadata": deepcopy(trips.metadata),
        "provenance": deepcopy(trips.provenance),
        "schema_version": trips.schema_version,
    }

### 0.2 - Fixture manual tripdataset_canonical_small

Qué construye: un TripDataset canónico pequeño, pero no pobre: 8 movements, varias columnas útiles, timestamps Tier 1, H3 válidos y mezcla de modos/propósitos.


In [2]:
# Puntos de apoyo para fabricar H3 válidos
points = {
    "A": (-33.4500, -70.6600),
    "B": (-33.4400, -70.6400),
    "C": (-33.4600, -70.6200),
    "D": (-33.4700, -70.6100),
    "E": (-33.4300, -70.6000),
    "F": (-33.4200, -70.5800),
    "G": (-33.4100, -70.5600),
    "H": (-33.4050, -70.5450),
    "Z1": (-33.3900, -70.5200),
    "Z2": (-33.3850, -70.5050),
}

cells8 = {k: h3_from_latlon(lat, lon, 8) for k, (lat, lon) in points.items()}
cells7 = {k: h3_to_parent(v, 7) for k, v in cells8.items()}

tripdataset_canonical_small = TripDataset(
    data=pd.DataFrame(
        [
            {
                "movement_id": "m0",
                "trip_id": "t0",
                "movement_seq": 0,
                "user_id": "u0",
                "origin_h3_index": cells8["A"],
                "destination_h3_index": cells8["B"],
                "origin_time_utc": "2026-01-01T08:05:00Z",
                "destination_time_utc": "2026-01-01T08:25:00Z",
                "mode": "bus",
                "purpose": "work",
                "day_type": "weekday",
                "time_period": "AM",
                "user_gender": "female",
                "trip_weight": 1.0,
                "origin_municipality": "Santiago",
                "destination_municipality": "Providencia",
            },
            {
                "movement_id": "m1",
                "trip_id": "t1",
                "movement_seq": 0,
                "user_id": "u1",
                "origin_h3_index": cells8["A"],
                "destination_h3_index": cells8["B"],
                "origin_time_utc": "2026-01-01T08:15:00Z",
                "destination_time_utc": "2026-01-01T08:40:00Z",
                "mode": "bus",
                "purpose": "work",
                "day_type": "weekday",
                "time_period": "AM",
                "user_gender": "male",
                "trip_weight": 1.2,
                "origin_municipality": "Santiago",
                "destination_municipality": "Providencia",
            },
            {
                "movement_id": "m2",
                "trip_id": "t2",
                "movement_seq": 0,
                "user_id": "u2",
                "origin_h3_index": cells8["A"],
                "destination_h3_index": cells8["C"],
                "origin_time_utc": "2026-01-01T09:10:00Z",
                "destination_time_utc": "2026-01-01T09:35:00Z",
                "mode": "metro",
                "purpose": "study",
                "day_type": "weekday",
                "time_period": "AM",
                "user_gender": "female",
                "trip_weight": 0.8,
                "origin_municipality": "Santiago",
                "destination_municipality": "Ñuñoa",
            },
            {
                "movement_id": "m3",
                "trip_id": "t3",
                "movement_seq": 0,
                "user_id": "u3",
                "origin_h3_index": cells8["D"],
                "destination_h3_index": cells8["E"],
                "origin_time_utc": "2026-01-01T10:05:00Z",
                "destination_time_utc": "2026-01-01T10:25:00Z",
                "mode": "walk",
                "purpose": "leisure",
                "day_type": "weekday",
                "time_period": "AM",
                "user_gender": "male",
                "trip_weight": 1.0,
                "origin_municipality": "Las Condes",
                "destination_municipality": "Las Condes",
            },
            {
                "movement_id": "m4",
                "trip_id": "t4",
                "movement_seq": 0,
                "user_id": "u4",
                "origin_h3_index": cells8["F"],
                "destination_h3_index": cells8["G"],
                "origin_time_utc": "2026-01-01T18:10:00Z",
                "destination_time_utc": "2026-01-01T18:40:00Z",
                "mode": "bike",
                "purpose": "leisure",
                "day_type": "weekday",
                "time_period": "PM",
                "user_gender": "female",
                "trip_weight": 1.0,
                "origin_municipality": "Providencia",
                "destination_municipality": "Ñuñoa",
            },
            {
                "movement_id": "m5",
                "trip_id": "t5",
                "movement_seq": 0,
                "user_id": "u5",
                "origin_h3_index": cells8["F"],
                "destination_h3_index": cells8["G"],
                "origin_time_utc": "2026-01-01T18:20:00Z",
                "destination_time_utc": "2026-01-01T18:45:00Z",
                "mode": "bike",
                "purpose": "leisure",
                "day_type": "weekday",
                "time_period": "PM",
                "user_gender": "male",
                "trip_weight": 1.5,
                "origin_municipality": "Providencia",
                "destination_municipality": "Ñuñoa",
            },
            {
                "movement_id": "m6",
                "trip_id": "t6",
                "movement_seq": 0,
                "user_id": "u6",
                "origin_h3_index": cells8["H"],
                "destination_h3_index": cells8["A"],
                "origin_time_utc": "2026-01-01T08:50:00Z",
                "destination_time_utc": "2026-01-01T09:15:00Z",
                "mode": "bus",
                "purpose": "work",
                "day_type": "weekday",
                "time_period": "AM",
                "user_gender": "female",
                "trip_weight": 1.0,
                "origin_municipality": "Santiago",
                "destination_municipality": "Santiago",
            },
            {
                "movement_id": "m7",
                "trip_id": "t7",
                "movement_seq": 0,
                "user_id": "u7",
                "origin_h3_index": cells8["B"],
                "destination_h3_index": cells8["A"],
                "origin_time_utc": "2026-01-01T07:55:00Z",
                "destination_time_utc": "2026-01-01T08:10:00Z",
                "mode": "bus",
                "purpose": "work",
                "day_type": "weekday",
                "time_period": "AM",
                "user_gender": "male",
                "trip_weight": 1.0,
                "origin_municipality": "Providencia",
                "destination_municipality": "Santiago",
            },
        ]
    ),
    schema=TripSchema(version="0.1.0", fields={}, required=[]),
    metadata={
        "dataset_id": "tripdataset_canonical_small",
        "is_validated": True,
        "temporal": {"tier": "tier_1"},
        "events": [
            {
                "op": "import_trips",
                "ts_utc": "2026-04-08T00:00:00Z",
                "parameters": {"profile": "synthetic_integration"},
                "summary": {"rows_out": 8},
            }
        ],
    },
    provenance={"source": "synthetic", "kind": "integration_test"},
)

### 0.3 - Fixtures manuales de flows

Qué construye:

- flowdataset_with_trip_links: camino directo feliz con auxiliar flow_to_trips
- flowdataset_small: reconstrucción desde trips, con un flow extra sin correspondencia
- flowdataset_source_trips_only: tercer fallback
- flowdataset_rollup_temporal: política clave de roll-up H3 + ventanas temporales

In [3]:
flowdataset_with_trip_links = FlowDataset(
    flows=pd.DataFrame(
        [
            {
                "flow_id": "f_ab_bus_work_h08",
                "origin_h3_index": cells8["A"],
                "destination_h3_index": cells8["B"],
                "window_start_utc": pd.Timestamp("2026-01-01 08:00:00"),
                "window_end_utc": pd.Timestamp("2026-01-01 09:00:00"),
                "mode": "bus",
                "purpose": "work",
                "flow_count": 2,
                "flow_value": 2.2,
            },
            {
                "flow_id": "f_ac_metro_study_h09",
                "origin_h3_index": cells8["A"],
                "destination_h3_index": cells8["C"],
                "window_start_utc": pd.Timestamp("2026-01-01 09:00:00"),
                "window_end_utc": pd.Timestamp("2026-01-01 10:00:00"),
                "mode": "metro",
                "purpose": "study",
                "flow_count": 1,
                "flow_value": 0.8,
            },
            {
                "flow_id": "f_fg_bike_leisure_h18",
                "origin_h3_index": cells8["F"],
                "destination_h3_index": cells8["G"],
                "window_start_utc": pd.Timestamp("2026-01-01 18:00:00"),
                "window_end_utc": pd.Timestamp("2026-01-01 19:00:00"),
                "mode": "bike",
                "purpose": "leisure",
                "flow_count": 2,
                "flow_value": 2.5,
            },
        ]
    ),
    flow_to_trips=pd.DataFrame(
        [
            {"flow_id": "f_ab_bus_work_h08", "movement_id": "m0"},
            {"flow_id": "f_ab_bus_work_h08", "movement_id": "m1"},
            {"flow_id": "f_ac_metro_study_h09", "movement_id": "m2"},
            {"flow_id": "f_fg_bike_leisure_h18", "movement_id": "m4"},
            {"flow_id": "f_fg_bike_leisure_h18", "movement_id": "m5"},
        ]
    ),
    aggregation_spec={
        "h3_resolution": 8,
        "group_by": ["mode", "purpose"],
        "time_aggregation": "hour",
        "time_basis": "origin",
        "effective_flow_keys": [
            "origin_h3_index",
            "destination_h3_index",
            "window_start_utc",
            "window_end_utc",
            "mode",
            "purpose",
        ],
    },
    source_trips=None,
    metadata={
        "dataset_id": "flowdataset_with_trip_links",
        "is_validated": False,
        "events": [
            {
                "op": "build_flows",
                "ts_utc": "2026-04-08T00:10:00Z",
                "parameters": {"keep_flow_to_trips": True},
                "summary": {"n_flows_out": 3},
            }
        ],
    },
    provenance={
        "derived_from": [{"type": "trips", "dataset_id": "tripdataset_canonical_small"}]
    },
)

flowdataset_small = FlowDataset(
    flows=pd.DataFrame(
        [
            {
                "flow_id": "f_ab_bus_work_h08",
                "origin_h3_index": cells8["A"],
                "destination_h3_index": cells8["B"],
                "window_start_utc": pd.Timestamp("2026-01-01 08:00:00"),
                "window_end_utc": pd.Timestamp("2026-01-01 09:00:00"),
                "mode": "bus",
                "purpose": "work",
                "flow_count": 2,
                "flow_value": 2.2,
            },
            {
                "flow_id": "f_ac_metro_study_h09",
                "origin_h3_index": cells8["A"],
                "destination_h3_index": cells8["C"],
                "window_start_utc": pd.Timestamp("2026-01-01 09:00:00"),
                "window_end_utc": pd.Timestamp("2026-01-01 10:00:00"),
                "mode": "metro",
                "purpose": "study",
                "flow_count": 1,
                "flow_value": 0.8,
            },
            {
                "flow_id": "f_fg_bike_leisure_h18",
                "origin_h3_index": cells8["F"],
                "destination_h3_index": cells8["G"],
                "window_start_utc": pd.Timestamp("2026-01-01 18:00:00"),
                "window_end_utc": pd.Timestamp("2026-01-01 19:00:00"),
                "mode": "bike",
                "purpose": "leisure",
                "flow_count": 2,
                "flow_value": 2.5,
            },
            {
                "flow_id": "f_unmatched_walk_other_h11",
                "origin_h3_index": cells8["Z1"],
                "destination_h3_index": cells8["Z2"],
                "window_start_utc": pd.Timestamp("2026-01-01 11:00:00"),
                "window_end_utc": pd.Timestamp("2026-01-01 12:00:00"),
                "mode": "walk",
                "purpose": "other",
                "flow_count": 7,
                "flow_value": 7.0,
            },
        ]
    ),
    flow_to_trips=None,
    aggregation_spec={
        "h3_resolution": 8,
        "group_by": ["mode", "purpose"],
        "time_aggregation": "hour",
        "time_basis": "origin",
        "effective_flow_keys": [
            "origin_h3_index",
            "destination_h3_index",
            "window_start_utc",
            "window_end_utc",
            "mode",
            "purpose",
        ],
    },
    source_trips=None,
    metadata={
        "dataset_id": "flowdataset_small",
        "is_validated": False,
        "events": [
            {
                "op": "build_flows",
                "ts_utc": "2026-04-08T00:20:00Z",
                "parameters": {"keep_flow_to_trips": False},
                "summary": {"n_flows_out": 4},
            }
        ],
    },
    provenance={
        "derived_from": [{"type": "trips", "dataset_id": "tripdataset_canonical_small"}]
    },
)

flowdataset_source_trips_only = FlowDataset(
    flows=flowdataset_small.flows.copy(deep=True),
    flow_to_trips=None,
    aggregation_spec=deepcopy(flowdataset_small.aggregation_spec),
    source_trips=tripdataset_canonical_small,
    metadata=deepcopy(flowdataset_small.metadata),
    provenance=deepcopy(flowdataset_small.provenance),
)

flowdataset_rollup_temporal = FlowDataset(
    flows=pd.DataFrame(
        [
            {
                "flow_id": "f_parent_ab_bus_work_d1",
                "origin_h3_index": cells7["A"],
                "destination_h3_index": cells7["B"],
                "window_start_utc": pd.Timestamp("2026-01-01 00:00:00"),
                "window_end_utc": pd.Timestamp("2026-01-02 00:00:00"),
                "mode": "bus",
                "purpose": "work",
                "flow_count": 2,
                "flow_value": 2.2,
            },
            {
                "flow_id": "f_parent_fg_bike_leisure_d1",
                "origin_h3_index": cells7["F"],
                "destination_h3_index": cells7["G"],
                "window_start_utc": pd.Timestamp("2026-01-01 00:00:00"),
                "window_end_utc": pd.Timestamp("2026-01-02 00:00:00"),
                "mode": "bike",
                "purpose": "leisure",
                "flow_count": 2,
                "flow_value": 2.5,
            },
        ]
    ),
    flow_to_trips=None,
    aggregation_spec={
        "h3_resolution": 7,
        "group_by": ["mode", "purpose"],
        "time_aggregation": "day",
        "time_basis": "origin",
        "effective_flow_keys": [
            "origin_h3_index",
            "destination_h3_index",
            "window_start_utc",
            "window_end_utc",
            "mode",
            "purpose",
        ],
    },
    source_trips=None,
    metadata={
        "dataset_id": "flowdataset_rollup_temporal",
        "is_validated": False,
        "events": [
            {
                "op": "build_flows",
                "ts_utc": "2026-04-08T00:30:00Z",
                "parameters": {"h3_resolution": 7, "time_aggregation": "day"},
                "summary": {"n_flows_out": 2},
            }
        ],
    },
    provenance={
        "derived_from": [{"type": "trips", "dataset_id": "tripdataset_canonical_small"}]
    },
)

print("Fixtures OK")
display(tripdataset_canonical_small.data.head())
display(flowdataset_with_trip_links.flows)

Fixtures OK


,movement_id,trip_id,movement_seq,user_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,day_type,time_period,user_gender,trip_weight,origin_municipality,destination_municipality
0,m0,t0,0,u0,88b2c55417fffff,88b2c554cdfffff,2026-01-01T08:05:00Z,2026-01-01T08:25:00Z,bus,work,weekday,AM,female,1.0,Santiago,Providencia
1,m1,t1,0,u1,88b2c55417fffff,88b2c554cdfffff,2026-01-01T08:15:00Z,2026-01-01T08:40:00Z,bus,work,weekday,AM,male,1.2,Santiago,Providencia
2,m2,t2,0,u2,88b2c55417fffff,88b2c554d5fffff,2026-01-01T09:10:00Z,2026-01-01T09:35:00Z,metro,study,weekday,AM,female,0.8,Santiago,Ñuñoa
3,m3,t3,0,u3,88b2c55499fffff,88b2c55695fffff,2026-01-01T10:05:00Z,2026-01-01T10:25:00Z,walk,leisure,weekday,AM,male,1.0,Las Condes,Las Condes
4,m4,t4,0,u4,88b2c519a5fffff,88b2c519a9fffff,2026-01-01T18:10:00Z,2026-01-01T18:40:00Z,bike,leisure,weekday,PM,female,1.0,Providencia,Ñuñoa


,flow_id,origin_h3_index,destination_h3_index,window_start_utc,window_end_utc,mode,purpose,flow_count,flow_value
0,f_ab_bus_work_h08,88b2c55417fffff,88b2c554cdfffff,2026-01-01 08:00:00,2026-01-01 09:00:00,bus,work,2,2.2
1,f_ac_metro_study_h09,88b2c55417fffff,88b2c554d5fffff,2026-01-01 09:00:00,2026-01-01 10:00:00,metro,study,1,0.8
2,f_fg_bike_leisure_h18,88b2c519a5fffff,88b2c519a9fffff,2026-01-01 18:00:00,2026-01-01 19:00:00,bike,leisure,2,2.5


## Bloque 1: Tests de integración de get_trips_from_flows

### 1.1 - Camino feliz usando flow_to_trips como fuente directa

Qué prueba: caso principal correcto. Debe usar la fuente prioritaria, devolver solo flow_id + movement_id, armar summary estable y no producir side effects. Esto cubre el happy path más limpio del contrato.


In [4]:
flows = deepcopy(flowdataset_with_trip_links)
flows_before = snapshot_flowdataset_state(flows)

flow_trip_correspondence_df, report = get_trips_from_flows(
    flows,
    max_issues=20,
)

assert flow_trip_correspondence_df.columns.tolist() == ["flow_id", "movement_id"]
assert list(flow_trip_correspondence_df.itertuples(index=False, name=None)) == [
    ("f_ab_bus_work_h08", "m0"),
    ("f_ab_bus_work_h08", "m1"),
    ("f_ac_metro_study_h09", "m2"),
    ("f_fg_bike_leisure_h18", "m4"),
    ("f_fg_bike_leisure_h18", "m5"),
]

assert report.ok is True
assert report.parameters == {
    "max_issues": 20,
    "used_source": "flow_to_trips",
    "reconstruction_attempted": False,
    "n_flows_input": 3,
    "n_trips_input": None,
}
assert report.summary == {
    "n_rows_out": 5,
    "n_unique_flows_out": 3,
    "n_unique_movements_out": 5,
    "n_unmatched_flows": 0,
    "n_unmatched_movements": 0,
}
assert report.issues == []

# OP-13 no debe mutar ni registrar evento
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
pd.testing.assert_frame_equal(flows.flow_to_trips, flows_before["flow_to_trips"])
assert flows.aggregation_spec == flows_before["aggregation_spec"]
assert flows.metadata == flows_before["metadata"]
assert flows.provenance == flows_before["provenance"]
assert flows.source_trips is flows_before["source_trips"]

display(flow_trip_correspondence_df)
display(report.summary)
display(report.parameters)

,flow_id,movement_id
0,f_ab_bus_work_h08,m0
1,f_ab_bus_work_h08,m1
2,f_ac_metro_study_h09,m2
3,f_fg_bike_leisure_h18,m4
4,f_fg_bike_leisure_h18,m5


{'n_rows_out': 5,
 'n_unique_flows_out': 3,
 'n_unique_movements_out': 5,
 'n_unmatched_flows': 0,
 'n_unmatched_movements': 0}

{'max_issues': 20,
 'used_source': 'flow_to_trips',
 'reconstruction_attempted': False,
 'n_flows_input': 3,
 'n_trips_input': None}

### 1.2 - Reconstrucción exacta desde trips_argument

Qué prueba: caso principal correcto por fallback, pero ahora reconstruyendo por OD + ventanas temporales + group_by. También verifica trip_id como enriquecimiento opcional y cobertura parcial cuando el universo de trips es más grande que el de flows. La reconstrucción debe usar exactamente las llaves efectivas de agregación.


In [6]:
trips = deepcopy(tripdataset_canonical_small)
flows = deepcopy(flowdataset_small)

flows_before = snapshot_flowdataset_state(flows)
trips_before = snapshot_tripdataset_state(trips)

flow_trip_correspondence_df, report = get_trips_from_flows(
    flows,
    trips=trips,
    max_issues=20,
)

assert flow_trip_correspondence_df.columns.tolist() == ["flow_id", "movement_id", "trip_id"]
assert list(flow_trip_correspondence_df.itertuples(index=False, name=None)) == [
    ("f_ab_bus_work_h08", "m0", "t0"),
    ("f_ab_bus_work_h08", "m1", "t1"),
    ("f_ac_metro_study_h09", "m2", "t2"),
    ("f_fg_bike_leisure_h18", "m4", "t4"),
    ("f_fg_bike_leisure_h18", "m5", "t5"),
]

assert report.ok is True
assert report.parameters == {
    "max_issues": 20,
    "used_source": "trips_argument",
    "reconstruction_attempted": True,
    "n_flows_input": 4,
    "n_trips_input": 8,
}
assert report.summary == {
    "n_rows_out": 5,
    "n_unique_flows_out": 3,
    "n_unique_movements_out": 5,
    "n_unmatched_flows": 1,
    "n_unmatched_movements": 3,
}
assert_issue_codes(
    report.issues,
    ["GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE"],
)

# Sin side effects sobre flows
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
assert flows.flow_to_trips is flows_before["flow_to_trips"]
assert flows.aggregation_spec == flows_before["aggregation_spec"]
assert flows.metadata == flows_before["metadata"]
assert flows.provenance == flows_before["provenance"]

# Sin side effects sobre trips
pd.testing.assert_frame_equal(trips.data, trips_before["data"])
assert trips.metadata == trips_before["metadata"]
assert trips.provenance == trips_before["provenance"]
assert trips.schema_version == trips_before["schema_version"]



print("\nFlujos, viajes originales y viajes obtenidos:")
display(flows.flows)
display(trips.data)
display(flow_trip_correspondence_df)
display(report.issues)
display(report.summary)


Flujos, viajes originales y viajes obtenidos:


,flow_id,origin_h3_index,destination_h3_index,window_start_utc,window_end_utc,mode,purpose,flow_count,flow_value
0,f_ab_bus_work_h08,88b2c55417fffff,88b2c554cdfffff,2026-01-01 08:00:00,2026-01-01 09:00:00,bus,work,2,2.2
1,f_ac_metro_study_h09,88b2c55417fffff,88b2c554d5fffff,2026-01-01 09:00:00,2026-01-01 10:00:00,metro,study,1,0.8
2,f_fg_bike_leisure_h18,88b2c519a5fffff,88b2c519a9fffff,2026-01-01 18:00:00,2026-01-01 19:00:00,bike,leisure,2,2.5
3,f_unmatched_walk_other_h11,88b2c519d9fffff,88b2c518e5fffff,2026-01-01 11:00:00,2026-01-01 12:00:00,walk,other,7,7.0


,movement_id,trip_id,movement_seq,user_id,origin_h3_index,destination_h3_index,origin_time_utc,destination_time_utc,mode,purpose,day_type,time_period,user_gender,trip_weight,origin_municipality,destination_municipality
0,m0,t0,0,u0,88b2c55417fffff,88b2c554cdfffff,2026-01-01T08:05:00Z,2026-01-01T08:25:00Z,bus,work,weekday,AM,female,1.0,Santiago,Providencia
1,m1,t1,0,u1,88b2c55417fffff,88b2c554cdfffff,2026-01-01T08:15:00Z,2026-01-01T08:40:00Z,bus,work,weekday,AM,male,1.2,Santiago,Providencia
2,m2,t2,0,u2,88b2c55417fffff,88b2c554d5fffff,2026-01-01T09:10:00Z,2026-01-01T09:35:00Z,metro,study,weekday,AM,female,0.8,Santiago,Ñuñoa
3,m3,t3,0,u3,88b2c55499fffff,88b2c55695fffff,2026-01-01T10:05:00Z,2026-01-01T10:25:00Z,walk,leisure,weekday,AM,male,1.0,Las Condes,Las Condes
4,m4,t4,0,u4,88b2c519a5fffff,88b2c519a9fffff,2026-01-01T18:10:00Z,2026-01-01T18:40:00Z,bike,leisure,weekday,PM,female,1.0,Providencia,Ñuñoa
5,m5,t5,0,u5,88b2c519a5fffff,88b2c519a9fffff,2026-01-01T18:20:00Z,2026-01-01T18:45:00Z,bike,leisure,weekday,PM,male,1.5,Providencia,Ñuñoa
6,m6,t6,0,u6,88b2c51981fffff,88b2c55417fffff,2026-01-01T08:50:00Z,2026-01-01T09:15:00Z,bus,work,weekday,AM,female,1.0,Santiago,Santiago
7,m7,t7,0,u7,88b2c554cdfffff,88b2c55417fffff,2026-01-01T07:55:00Z,2026-01-01T08:10:00Z,bus,work,weekday,AM,male,1.0,Providencia,Santiago


,flow_id,movement_id,trip_id
0,f_ab_bus_work_h08,m0,t0
1,f_ab_bus_work_h08,m1,t1
2,f_ac_metro_study_h09,m2,t2
3,f_fg_bike_leisure_h18,m4,t4
4,f_fg_bike_leisure_h18,m5,t5


[Issue(level='warning', code='GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE', message='La correspondencia flujo-viajes quedó parcial: 3 movements y 1 flows quedaron sin correspondencia.', field=None, source_field=None, row_count=None, details={'max_issues': 20, 'n_flows_input': 4, 'n_trips_input': 8, 'used_source': 'trips_argument', 'reconstruction_attempted': True, 'source': 'trips_argument', 'join_key_columns': ['origin_h3_index', 'destination_h3_index', 'window_start_utc', 'window_end_utc', 'mode', 'purpose'], 'group_by': ['mode', 'purpose'], 'window_columns': ['window_start_utc', 'window_end_utc'], 'n_rows_out': 5, 'n_unmatched_movements': 3, 'n_unmatched_flows': 1, 'example_values': {'flow_id_sample_unmatched': ['f_unmatched_walk_other_h11'], 'movement_id_sample_unmatched': ['m7', 'm3', 'm6']}, 'reason': 'partial_coverage', 'action': 'return_partial_correspondence'})]

{'n_rows_out': 5,
 'n_unique_flows_out': 3,
 'n_unique_movements_out': 5,
 'n_unmatched_flows': 1,
 'n_unmatched_movements': 3}

### 1.3 - Degradación con warning: flow_to_trips no usable y caída a trips_argument

Qué prueba: caso degradado con warning. La fuente prioritaria existe pero es inválida, así que la operación debe avisarlo, usar trips_argument y seguir siendo útil. Esta es la mejor prueba integrada de la política de prioridad de fuentes más warning por degradación controlada.


In [7]:
trips = deepcopy(tripdataset_canonical_small)
flows = deepcopy(flowdataset_small)

# Se agrega un auxiliar mal formado para forzar la degradación desde la fuente prioritaria.
flows.flow_to_trips = pd.DataFrame(
    [
        {"flow_id": "f_ab_bus_work_h08"},
        {"flow_id": "f_ac_metro_study_h09"},
    ]
)

flows_before = snapshot_flowdataset_state(flows)
trips_before = snapshot_tripdataset_state(trips)

flow_trip_correspondence_df, report = get_trips_from_flows(
    flows,
    trips=trips,
    max_issues=20,
)

codes = [issue.code for issue in report.issues]

assert report.ok is True
assert report.parameters["used_source"] == "trips_argument"
assert report.parameters["reconstruction_attempted"] is True
assert report.parameters["n_trips_input"] == 8

assert "GET_TRIPS_FROM_FLOWS.SOURCE.PREFERRED_SOURCE_UNUSABLE" in codes
assert "GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE" in codes

assert flow_trip_correspondence_df.columns.tolist() == ["flow_id", "movement_id", "trip_id"]
assert len(flow_trip_correspondence_df) == 5

# Sin side effects
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
pd.testing.assert_frame_equal(flows.flow_to_trips, flows_before["flow_to_trips"])
assert flows.metadata == flows_before["metadata"]

pd.testing.assert_frame_equal(trips.data, trips_before["data"])
assert trips.metadata == trips_before["metadata"]

display(flow_trip_correspondence_df)
display(report.issues)
display(report.parameters)

,flow_id,movement_id,trip_id
0,f_ab_bus_work_h08,m0,t0
1,f_ab_bus_work_h08,m1,t1
2,f_ac_metro_study_h09,m2,t2
3,f_fg_bike_leisure_h18,m4,t4
4,f_fg_bike_leisure_h18,m5,t5


[Issue(level='warning', code='GET_TRIPS_FROM_FLOWS.SOURCE.PREFERRED_SOURCE_UNUSABLE', message="La fuente prioritaria 'flow_to_trips' no fue usable; la operación continuó con 'trips_argument'.", field=None, source_field=None, row_count=None, details={'max_issues': 20, 'n_flows_input': 4, 'n_trips_input': None, 'used_source': 'trips_argument', 'reconstruction_attempted': False, 'preferred_source': 'flow_to_trips', 'checked_sources': ['flow_to_trips', 'trips_argument'], 'source_failures': {'flow_to_trips': {'reason': 'missing_required_columns', 'required_columns': ['flow_id', 'movement_id'], 'missing_columns': ['movement_id']}}, 'required_columns': ['flow_id', 'movement_id'], 'missing_columns': ['movement_id'], 'reason': 'preferred_source_unusable', 'action': 'use_fallback'}),
 Issue(level='warning', code='GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE', message='La correspondencia flujo-viajes quedó parcial: 3 movements y 1 flows quedaron sin correspondencia.', field=None, source_field=Non

{'max_issues': 20,
 'used_source': 'trips_argument',
 'reconstruction_attempted': True,
 'n_flows_input': 4,
 'n_trips_input': 8}

### 1.4 - Tercer fallback: uso de flows.source_trips

Qué prueba: política clave de prioridad cuando no viene trips explícito pero sí existe flows.source_trips en memoria. Esto además refleja la decisión del cierre concreto de que este fallback es válido solo para datasets vivos en memoria.


In [10]:
flows = deepcopy(flowdataset_source_trips_only)
flows_before = snapshot_flowdataset_state(flows)

flow_trip_correspondence_df, report = get_trips_from_flows(
    flows,
    trips=None,
    max_issues=20,
)

assert report.ok is True
assert report.parameters == {
    "max_issues": 20,
    "used_source": "flows.source_trips",
    "reconstruction_attempted": True,
    "n_flows_input": 4,
    "n_trips_input": 8,
}
assert report.summary == {
    "n_rows_out": 5,
    "n_unique_flows_out": 3,
    "n_unique_movements_out": 5,
    "n_unmatched_flows": 1,
    "n_unmatched_movements": 3,
}
assert_issue_codes(
    report.issues,
    ["GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE"],
)

assert flow_trip_correspondence_df.columns.tolist() == ["flow_id", "movement_id", "trip_id"]

# OP-13 no debe tocar metadata ni eventos
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
assert flows.flow_to_trips is flows_before["flow_to_trips"]
assert flows.metadata == flows_before["metadata"]
assert flows.provenance == flows_before["provenance"]

display(flow_trip_correspondence_df)
display(report.summary)
display(report.parameters)

,flow_id,movement_id,trip_id
0,f_ab_bus_work_h08,m0,t0
1,f_ab_bus_work_h08,m1,t1
2,f_ac_metro_study_h09,m2,t2
3,f_fg_bike_leisure_h18,m4,t4
4,f_fg_bike_leisure_h18,m5,t5


{'n_rows_out': 5,
 'n_unique_flows_out': 3,
 'n_unique_movements_out': 5,
 'n_unmatched_flows': 1,
 'n_unmatched_movements': 3}

{'max_issues': 20,
 'used_source': 'flows.source_trips',
 'reconstruction_attempted': True,
 'n_flows_input': 4,
 'n_trips_input': 8}

### 1.5 - Fatal de precondición: falta flow_id en flows.flows

Qué prueba: caso fatal/precondición. Si el contrato mínimo del FlowDataset está roto, la operación debe abortar antes de resolver fuentes o intentar reconstruir. Esto cubre un borde crítico del contrato observable.

In [11]:
flows = deepcopy(flowdataset_with_trip_links)
flows.flows = flows.flows.drop(columns=["flow_id"]).copy()

flows_before = snapshot_flowdataset_state(flows)

raised = None
try:
    get_trips_from_flows(
        flows,
        max_issues=20,
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, PylondrinaError)
assert raised.issue is not None
assert raised.issue.code == "GET_TRIPS_FROM_FLOWS.DATA.MISSING_FLOW_ID"

# No debe haber mutación ni siquiera tras el fallo.
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
assert flows.metadata == flows_before["metadata"]
assert flows.provenance == flows_before["provenance"]

display(raised.issue)

Issue(level='error', code='GET_TRIPS_FROM_FLOWS.DATA.MISSING_FLOW_ID', message="El FlowDataset no contiene la columna requerida 'flow_id' en `flows.flows`.", field=None, source_field=None, row_count=None, details={'max_issues': 20, 'n_flows_input': 3, 'n_trips_input': None, 'used_source': None, 'reconstruction_attempted': False, 'field': 'flow_id', 'available_fields_sample': ['origin_h3_index', 'destination_h3_index', 'window_start_utc', 'window_end_utc', 'mode', 'purpose', 'flow_count', 'flow_value'], 'available_fields_total': 8, 'reason': 'missing_required_column', 'action': 'abort'})

### 1.6 - Fatal operativo: no existe ninguna fuente usable

Qué prueba: caso fatal muy propio de OP-13. No hay flow_to_trips, no se entrega trips, y tampoco existe flows.source_trips. La operación debe abortar, no retornar una tabla vacía.


In [12]:
flows = deepcopy(flowdataset_small)
flows.flow_to_trips = None
flows.source_trips = None

flows_before = snapshot_flowdataset_state(flows)

raised = None
try:
    get_trips_from_flows(
        flows,
        trips=None,
        max_issues=20,
    )
except Exception as e:
    raised = e

assert raised is not None
assert isinstance(raised, PylondrinaError)
assert raised.issue is not None
assert raised.issue.code == "GET_TRIPS_FROM_FLOWS.SOURCE.NO_USABLE_SOURCE"

# Tampoco aquí debe haber side effects.
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
assert flows.metadata == flows_before["metadata"]
assert flows.provenance == flows_before["provenance"]

display(raised.issue)

Issue(level='error', code='GET_TRIPS_FROM_FLOWS.SOURCE.NO_USABLE_SOURCE', message='No existe ninguna fuente usable para construir la correspondencia flujo-viajes.', field=None, source_field=None, row_count=None, details={'max_issues': 20, 'n_flows_input': 4, 'n_trips_input': None, 'used_source': None, 'reconstruction_attempted': False, 'preferred_source': 'flow_to_trips', 'checked_sources': ['flow_to_trips', 'trips_argument', 'flows.source_trips'], 'source_failures': {'flow_to_trips': {'reason': 'missing', 'required_columns': ['flow_id', 'movement_id'], 'missing_columns': ['flow_id', 'movement_id']}, 'trips_argument': {'reason': 'missing', 'required_columns': ['movement_id', 'origin_h3_index', 'destination_h3_index'], 'missing_columns': ['movement_id', 'origin_h3_index', 'destination_h3_index']}, 'flows.source_trips': {'reason': 'missing', 'required_columns': ['movement_id', 'origin_h3_index', 'destination_h3_index'], 'missing_columns': ['movement_id', 'origin_h3_index', 'destination_h

### 1.7 - Política clave: reconstrucción con roll-up H3 y ventanas temporales diarias

Qué prueba: soporte real de una de las políticas más delicadas de OP-13: si los flows fueron construidos a una resolución H3 más gruesa y con agregación temporal, la reconstrucción debe reproducir exactamente ese contrato. Aquí se prueba roll-up a resolución 7 y ventanas diarias.


In [13]:

trips = deepcopy(tripdataset_canonical_small)
flows = deepcopy(flowdataset_rollup_temporal)

flows_before = snapshot_flowdataset_state(flows)
trips_before = snapshot_tripdataset_state(trips)

flow_trip_correspondence_df, report = get_trips_from_flows(
    flows,
    trips=trips,
    max_issues=20,
)

assert flow_trip_correspondence_df.columns.tolist() == ["flow_id", "movement_id", "trip_id"]
assert list(flow_trip_correspondence_df.itertuples(index=False, name=None)) == [
    ("f_parent_ab_bus_work_d1", "m0", "t0"),
    ("f_parent_ab_bus_work_d1", "m1", "t1"),
    ("f_parent_fg_bike_leisure_d1", "m4", "t4"),
    ("f_parent_fg_bike_leisure_d1", "m5", "t5"),
]

assert report.ok is True
assert report.parameters == {
    "max_issues": 20,
    "used_source": "trips_argument",
    "reconstruction_attempted": True,
    "n_flows_input": 2,
    "n_trips_input": 8,
}
assert report.summary == {
    "n_rows_out": 4,
    "n_unique_flows_out": 2,
    "n_unique_movements_out": 4,
    "n_unmatched_flows": 0,
    "n_unmatched_movements": 4,
}
assert_issue_codes(
    report.issues,
    ["GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE"],
)

# Sin side effects
pd.testing.assert_frame_equal(flows.flows, flows_before["flows"])
assert flows.metadata == flows_before["metadata"]
pd.testing.assert_frame_equal(trips.data, trips_before["data"])
assert trips.metadata == trips_before["metadata"]

display(flow_trip_correspondence_df)
display(report.issues)
display(report.summary)

,flow_id,movement_id,trip_id
0,f_parent_ab_bus_work_d1,m0,t0
1,f_parent_ab_bus_work_d1,m1,t1
2,f_parent_fg_bike_leisure_d1,m4,t4
3,f_parent_fg_bike_leisure_d1,m5,t5


[Issue(level='warning', code='GET_TRIPS_FROM_FLOWS.OUTPUT.PARTIAL_COVERAGE', message='La correspondencia flujo-viajes quedó parcial: 4 movements y 0 flows quedaron sin correspondencia.', field=None, source_field=None, row_count=None, details={'max_issues': 20, 'n_flows_input': 2, 'n_trips_input': 8, 'used_source': 'trips_argument', 'reconstruction_attempted': True, 'source': 'trips_argument', 'join_key_columns': ['origin_h3_index', 'destination_h3_index', 'window_start_utc', 'window_end_utc', 'mode', 'purpose'], 'group_by': ['mode', 'purpose'], 'window_columns': ['window_start_utc', 'window_end_utc'], 'n_rows_out': 4, 'n_unmatched_movements': 4, 'n_unmatched_flows': 0, 'example_values': {'flow_id_sample_unmatched': [], 'movement_id_sample_unmatched': ['m2', 'm7', 'm3', 'm6']}, 'reason': 'partial_coverage', 'action': 'return_partial_correspondence'})]

{'n_rows_out': 4,
 'n_unique_flows_out': 2,
 'n_unique_movements_out': 4,
 'n_unmatched_flows': 0,
 'n_unmatched_movements': 4}